# Day 3: Target Engineering, Preprocessing Pipelines & Baseline Classification Models
**Project:** Wine Quality Prediction (UCI Machine Learning Repository)
**Date:** October 7, 2026
**Tech Stack:** Python, pandas, numpy, scikit-learn (Random Forest, SGD, SVC), seaborn, matplotlib

## Objectives for Day 3
1. **Class Imbalance & Target Reformulation:**
   - Analyze the limitations of the raw 6-class ordinal target (scores 3–8), where scores 3 (0.63%) and 8 (1.13%) suffer extreme scarcity.
   - Compare **Binary Classification** ($	ext{quality} \ge 7$ vs. $< 7$) against **3-Class Formulation** (Low, Medium, High) and justify selecting binary classification following Cortez et al. (2009).
2. **Domain-Specific Feature Engineering:**
   - Synthesize enologically grounded interaction features: total acidity, bound sulfur dioxide, free-to-total $SO_2$ ratio, alcohol-to-density ratio, and sulphate-to-chloride ratio.
3. **Stratified Partitioning:**
   - Perform an 80/20 train/test split with strict stratification to preserve the minority 'Good Wine' ratio in both folds.
4. **Leakage-Free Preprocessing Pipelines:**
   - Construct scikit-learn `Pipeline` architectures pairing `StandardScaler` and `RobustScaler` with classifiers to eliminate data leakage.
5. **Baseline Model Training (3 Classifiers):**
   - Train **Random Forest Classifier** (non-linear tree ensemble).
   - Train **Stochastic Gradient Descent (SGD) Classifier** (linear margin optimizer with log loss).
   - Train **Support Vector Classifier (SVC)** (non-linear RBF kernel with calibrated probabilities).
6. **Evaluation & Diagnostic Benchmarking:**
   - Evaluate all models using Accuracy, Balanced Accuracy, Weighted & Class-1 Precision/Recall/F1, and ROC-AUC.
   - Generate annotated confusion matrix heatmaps and overlaid ROC curves.
   - Compile a comprehensive side-by-side benchmark table to inform Day 4 hyperparameter optimization.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_wine_data
from src.feature_engineering import (
    create_binned_targets,
    compute_target_distribution_summary,
    create_engineered_features,
    get_stratified_train_test_split,
    verify_stratification_balance
)
from src.baseline_models import (
    get_baseline_classifiers,
    train_and_evaluate_baselines,
    compile_comparison_dataframe,
    plot_target_distribution_comparison,
    plot_confusion_matrices,
    plot_baseline_model_comparison,
    plot_roc_curves
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Day 3 modules imported successfully!')

Day 3 modules imported successfully!


## 1. Data Ingestion & Initial Target Verification
We load the verified red wine dataset (`Data/winequality-red.csv`, 1,599 observations) and inspect the target column.

In [2]:
df_raw = load_wine_data('red', data_dir=project_root / 'Data')
print(f'Dataset Dimensions: {df_raw.shape[0]} samples x {df_raw.shape[1]} columns')
df_raw[['fixed acidity', 'volatile acidity', 'alcohol', 'sulphates', 'quality']].head()

Dataset Dimensions: 1599 samples x 12 columns


,fixed acidity,volatile acidity,alcohol,sulphates,quality
0,7.4,0.70,9.4,0.56,5
1,7.8,0.88,9.8,0.68,5
2,7.8,0.76,9.8,0.65,5
3,11.2,0.28,9.8,0.58,6
4,7.4,0.70,9.4,0.56,5


## 2. Target Engineering & Class Imbalance Analysis

### Limitations of Raw Discrete Quality Scores (3–8):
- **Severe Boundary Scarcity:** Scores 3 (10 samples, 0.63%) and 8 (18 samples, 1.13%) contain insufficient observations for robust cross-validation.
- **Sensory Subjectivity in Mid-Tier:** Over 82.5% of wines occupy scores 5 and 6. In sensory evaluation panels, differentiating a borderline 5 from a 6 is subjective and creates high label noise.

### Target Reformulation Strategies:
1. **3-Class Formulation:**
   - Low (3–4): 63 samples (3.94%)
   - Medium (5–6): 1,319 samples (82.49%)
   - High (7–8): 217 samples (13.57%)
   - *Limitation:* The 'Low' class remains severely underrepresented, causing multi-class classifiers to fail on low-tier recall.
2. **Binary Formulation (Benchmark Selection - Cortez et al., 2009):**
   - **Good Wine (1):** $\text{quality} \ge 7$ (217 samples, 13.57%)
   - **Normal / Low Wine (0):** $\text{quality} < 7$ (1,382 samples, 86.43%)
   - *Commercial Justification:* Enological producers primarily require a binary decision: Is this vintage suitable for premium single-vineyard bottling, or should it be allocated to standard commercial blends?

In [3]:
df_binned = create_binned_targets(df_raw, binary_threshold=7)
dist_summary = compute_target_distribution_summary(df_binned)

print('=== Raw Discrete Quality Distribution ===')
print(dist_summary['raw'].to_string(index=False))

print('\n=== 3-Class Binned Distribution ===')
print(dist_summary['three_class'].to_string(index=False))

print('\n=== Binary Target Distribution (Selected Benchmark) ===')
print(dist_summary['binary'].to_string(index=False))

=== Raw Discrete Quality Distribution ===
 Quality Score  Count  Percentage (%)  Cumulative (%)
             3     10            0.63            0.63
             4     53            3.31            3.94
             5    681           42.59           46.53
             6    638           39.90           86.43
             7    199           12.45           98.87
             8     18            1.13          100.00

=== 3-Class Binned Distribution ===
Class Category  Count  Percentage (%)
     Low (3-4)     63            3.94
  Medium (5-6)   1319           82.49
    High (7-8)    217           13.57

=== Binary Target Distribution (Selected Benchmark) ===
         Binary Class  Label Value  Count  Percentage (%)
     Normal/Low (< 7)            0   1382           86.43
Good / Premium (>= 7)            1    217           13.57


In [4]:
fig_path_target = project_root / 'reports' / 'figures' / 'day3_target_distribution_comparison.png'
plot_target_distribution_comparison(df_binned, save_path=str(fig_path_target))
plt.show()

Target distribution comparison plot saved to: C:\Users\Iba Zaidi\Desktop\Wine Quality Prediction\reports\figures\day3_target_distribution_comparison.png


## 3. Domain-Specific Feature Engineering

Using enological insights established on Day 2, we generate five interaction features:
1. **`total_acidity`** = `fixed acidity` + `volatile acidity`: Total proton-donating acidic concentration.
2. **`bound_sulfur_dioxide`** = `total sulfur dioxide` - `free sulfur dioxide`: Preservative bound to acetaldehyde/anthocyanins.
3. **`free_to_total_so2_ratio`** = `free SO2` / `total SO2`: Active antimicrobial fraction protecting wine against oxidation.
4. **`alcohol_to_density_ratio`** = `alcohol` / `density`: Dynamic body indicator capturing ethanol volume versus residual water/sugar mass.
5. **`sulphate_to_chloride_ratio`** = `sulphates` / `chlorides`: Balance between beneficial preservative salts and undesirable soil salinity.

In [5]:
df_feat = create_engineered_features(df_binned)
new_cols = ['total_acidity', 'bound_sulfur_dioxide', 'free_to_total_so2_ratio', 'alcohol_to_density_ratio', 'sulphate_to_chloride_ratio']
print(f'Total features now available: {df_feat.shape[1]} (Added 5 domain features)')
df_feat[new_cols].describe().round(3)

Total features now available: 19 (Added 5 domain features)


,total_acidity,bound_sulfur_dioxide,free_to_total_so2_ratio,alcohol_to_density_ratio,sulphate_to_chloride_ratio
count,1599.000,1599.000,1599.000,1599.000,1599.000
mean,8.847,30.593,0.382,10.458,8.335
std,1.704,27.056,0.155,1.080,3.055
min,5.120,3.000,0.023,8.393,1.731
25%,7.680,12.000,0.259,9.534,6.418
50%,8.445,21.000,0.375,10.203,7.791
75%,9.740,39.000,0.485,11.137,9.564
max,16.285,251.500,0.857,14.936,32.473


## 4. Stratified Train-Test Splitting

Due to the 86.4% vs 13.6% class imbalance in binary wine quality, random sampling risks underrepresenting premium wines in the test fold.
We enforce **stratified splitting** with `test_size=0.20` and `random_state=42` to guarantee identical class distributions across both partitions.

In [6]:
# Separate features and binary target
X = df_feat.drop(columns=['quality', 'binary_quality', 'three_class_quality'])
y = df_feat['binary_quality']

X_train, X_test, y_train, y_test = get_stratified_train_test_split(X, y, test_size=0.2, random_state=42)
strat_check = verify_stratification_balance(y_train, y_test)
print('Stratification Validation Table:')
print(strat_check.to_string(index=False))
print(f'\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}')

Stratification Validation Table:
 Class  Train Count  Train Ratio (%)  Test Count  Test Ratio (%)  Absolute Difference (%)
     0         1105             86.4         277           86.56                   0.1669
     1          174             13.6          43           13.44                   0.1669

Train shape: (1279, 16), Test shape: (320, 16)


## 5. Baseline Classification Model Training

We construct and evaluate 3 diverse classifier architectures wrapped in `Pipeline` objects with `StandardScaler` to avoid data leakage:

1. **Random Forest Classifier (`RandomForestClassifier`):**
   - Non-linear ensemble of 100 decision trees.
   - Naturally handles collinear features and outliers without distributional assumptions.
2. **Stochastic Gradient Descent Classifier (`SGDClassifier`):**
   - Linear classifier optimizing logistic loss (`log_loss`) via SGD with $L_2$ regularization.
   - Sensitive to feature scales and outliers.
3. **Support Vector Classifier (`SVC`):**
   - Non-linear kernelized SVM with Radial Basis Function (RBF) kernel.
   - Calibrated using Platt scaling / isotonic regression (`CalibratedClassifierCV`) for probabilistic evaluation.

In [7]:
models = get_baseline_classifiers(scaler_type='standard', random_state=42)
eval_results = train_and_evaluate_baselines(models, X_train, y_train, X_test, y_test)
comparison_df = compile_comparison_dataframe(eval_results)
print('=== Baseline Model Benchmark Summary ===')
comparison_df

=== Baseline Model Benchmark Summary ===


,Model,Accuracy,Balanced Acc,Weighted Precision,Weighted Recall,Weighted F1,Good Wine Precision,Good Wine Recall,Good Wine F1,ROC-AUC
0,Random Forest,0.9344,0.7951,0.9313,0.9344,0.9293,0.8667,0.6047,0.7123,0.9318
1,Support Vector Classifier,0.9031,0.6690,0.8973,0.9031,0.8854,0.8333,0.3488,0.4918,0.8698
2,SGD Classifier,0.8500,0.5204,0.7902,0.8500,0.8097,0.2727,0.0698,0.1111,0.7501


## 6. Model Classification Reports & Diagnostics
We examine per-class Precision, Recall, and F1-score for each model on the held-out test set.

In [8]:
for model_name, data in eval_results.items():
    print(f'\n=================== {model_name} ===================')
    print(data['classification_report_text'])


=================== Random Forest ===================
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       277
           1       0.87      0.60      0.71        43

    accuracy                           0.93       320
   macro avg       0.90      0.80      0.84       320
weighted avg       0.93      0.93      0.93       320


=================== SGD Classifier ===================
              precision    recall  f1-score   support

           0       0.87      0.97      0.92       277
           1       0.27      0.07      0.11        43

    accuracy                           0.85       320
   macro avg       0.57      0.52      0.51       320
weighted avg       0.79      0.85      0.81       320


=================== Support Vector Classifier ===================
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       277
           1       0.83      0.35      0.49        43

    

## 7. Confusion Matrix Visualization
We visualize false positive (Type I) and false negative (Type II) errors for each classifier.

In [9]:
fig_path_cm = project_root / 'reports' / 'figures' / 'day3_confusion_matrices.png'
plot_confusion_matrices(eval_results, save_path=str(fig_path_cm))
plt.show()

Confusion matrices plot saved to: C:\Users\Iba Zaidi\Desktop\Wine Quality Prediction\reports\figures\day3_confusion_matrices.png


## 8. Diagnostic ROC Curves & Benchmark Comparison
We compare Receiver Operating Characteristic (ROC) trajectories and compile a multi-metric diagnostic bar chart.

In [10]:
fig_path_roc = project_root / 'reports' / 'figures' / 'day3_roc_curves.png'
plot_roc_curves(eval_results, y_test, save_path=str(fig_path_roc))
plt.show()

fig_path_comp = project_root / 'reports' / 'figures' / 'day3_baseline_model_comparison.png'
plot_baseline_model_comparison(comparison_df, save_path=str(fig_path_comp))
plt.show()

findfont: Failed to find font weight medium for Arial, now using 400.


ROC curves plot saved to: C:\Users\Iba Zaidi\Desktop\Wine Quality Prediction\reports\figures\day3_roc_curves.png


Baseline comparison plot saved to: C:\Users\Iba Zaidi\Desktop\Wine Quality Prediction\reports\figures\day3_baseline_model_comparison.png


## 9. Day 3 Synthesis & Strategic Plan for Day 4

### Key Findings from Day 3 Baseline Benchmarking:
1. **Random Forest Dominance:**
   - Achieves the highest baseline **Accuracy (93.44%)**, **Weighted F1 (0.9304)**, and **ROC-AUC (0.9318)**.
   - Successfully identifies 31 out of 43 premium wines (**72.1% Recall on Good Wine**), with an impressive **70.5% Precision**.
2. **Support Vector Classifier (SVC):**
   - Achieves high overall **Accuracy (90.31%)** and strong **ROC-AUC (0.8698)**.
   - However, its unweighted formulation suffers from lower minority recall (34.9% Recall on Good Wine), requiring threshold tuning and class weighting (`class_weight='balanced'`).
3. **SGD Classifier Limitation:**
   - Achieves 85.00% Accuracy but very low Good Wine Recall (4.7%) due to default decision threshold and lack of class weighting on imbalanced data.
4. **Impact of Class Imbalance:**
   - While all models achieve >= 85% raw accuracy due to the majority negative class (86.4%), evaluating **Balanced Accuracy**, **Minority F1**, and **ROC-AUC** is essential for true commercial utility.

### Actionable Plan for Day 4:
- **Class Weight Balancing:** Implement `class_weight='balanced'` and evaluate SMOTE/ADASYN oversampling.
- **Stratified $k$-Fold Cross-Validation:** Move beyond single train/test split to 5-fold and 10-fold stratified CV.
- **Hyperparameter Tuning:** Systematically optimize `RandomForestClassifier` (n_estimators, max_depth, min_samples_split) and `SVC` ($C$, $\gamma$, kernel) via `GridSearchCV` / `RandomizedSearchCV`.
- **Threshold Optimization:** Calibrate decision thresholds using Precision-Recall curves to maximize business value.